Code to set path root

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.3
    ),
    transforms.RandomHorizontalFlip(p=0.35),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2)
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 3 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

In [ ]:
model.add(
    # First Layer
    nn.Conv2d(3, 12, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(12),
    nn.MaxPool2d(2, 2),

    # Second Layer
    nn.Conv2d(12, 24, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(24),
    nn.MaxPool2d(2, 2),

    # Third Layer
    nn.Conv2d(24, 48, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(48),
    nn.MaxPool2d(2, 2),

    # Flatten
    nn.Flatten(),

    # Fully connected layer (for 64x64 input)
    nn.Linear(48 * 8 * 8, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
)

### Use Trainer to train and check validations

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment1/",
    save_checkpoints=1,
    print_every=10
    )

In [ ]:
history = trainer.fit(100)

In [ ]:
df = plot_training_metrics(history)

In [ ]:
filtered_df = df[
    (df["val_accuracy"] >= 0.85) &
    (df["val_f1"] >= 0.85)
].to_dict()

high_accuracy_df = plot_training_metrics(filtered_df)

In [ ]:
high_accuracy_df

### Best Performing Model at Epoch 58
- epoch: 58
- train_loss: 0.066941
- val_loss: 0.773200
- train_accuracy: 0.983459
- train_precision: 0.983428
- train_recall: 0.983400
- train_f1: 0.983411
- val_accuracy: 0.871681
- val_precision: 0.872816
- val_recall: 0.871766
- val_f1: 0.872153